In [ ]:
%matplotlib notebook

import pickle
import pandas as pd
import matplotlib.pyplot as plt
from bee_simulation.model import OpenCoverModel
from tqdm import tqdm
import numpy as np
# ------------------------------
# User-defined simulation inputs
# ------------------------------

behavior = "resource_aware"  # Choose from: pure_levy, truncated_levy, composite_levy, biased_composite, my_behavior
seed_value = 0            # Random seed
steps = 60 * 60           # Number of simulation steps (e.g., 10 minutes)
n_agents = 60             # Number of bees
landing_prob = 1.0        # Bee landing probability

# ------------------------------
# Initialize and run the model
# ------------------------------

print(f"Running simulation for behavior: {behavior} with seed {seed_value}")

# 1. Initialize model
model = BlankModel(
    n_agents=n_agents,
    bee_behavior=behavior,
    landing_prob=landing_prob,
    seed=seed_value
)

# 2. Step the model
for _ in tqdm(range(steps)):
# for _ in range(steps):
    model.step()

# ------------------------------
# Save model data (DataCollector results)
# ------------------------------

df = model.datacollector.get_agent_vars_dataframe()
output_filename = f"sim_results_{behavior}_seed_{seed_value}_blank.pkl"

with open(output_filename, "wb") as f:
    pickle.dump(df, f)

print(f"Simulation complete. DataCollector results saved to {output_filename}")

In [ ]:
# Collecting sampled turning angles
all_angles = []
for agent in model.agents:
    if hasattr(agent, "turning_angles"):
        all_angles.extend(agent.turning_angles)

angle_filename = f"bee_turning_angles_{behavior}_seed_{seed_value}.pkl"
with open(angle_filename, "wb") as f:
    pickle.dump(all_angles, f)

print(f"Turning angles saved to {angle_filename}")

# Collecting sampled distances
all_distances = []
for agent in model.agents:
    if hasattr(agent, "jump_distances"):
        all_distances.extend(agent.jump_distances)
distance_filename = f"bee_jump_distances_{behavior}_seed_{seed_value}.pkl"
with open(distance_filename, "wb") as f:
    pickle.dump(all_distances, f)

print(f"Jumping distance saved to {distance_filename}")

## Turning angles verification (Blank)

In [ ]:
%matplotlib inline
# Define PDF
x_discrete = np.array([-180, -135, -90, -45, 0, 45, 90, 135, 180])
y_discrete = np.array([0.06, 0.07, 0.13, 0.17, 0.20, 0.17, 0.13, 0.07, 0.06])

x_fine = np.linspace(-180, 180, 1000)
pdf_fine = np.interp(x_fine, x_discrete, y_discrete)
dx_pdf = x_fine[1] - x_fine[0]
pdf_fine /= np.sum(pdf_fine * dx_pdf)

# Plot histogram
n, bins, patches = plt.hist(
    all_angles,
    bins=36,
    range=(-180, 180),
    density=True,
    alpha=0.5,
    color="orange"
)
patches[0].set_label("Sampled Angles")  # Set label only once

# Plot target PDF
plt.plot(
    x_fine,
    pdf_fine,
    label="Biased Turning Angles",
    color="blue",
    linewidth=2
)

plt.title("Sampled vs Target PDF of Free-Flight Turning Angles")
plt.xlabel("Turning Angle (degrees)")
plt.ylabel("Density")
plt.legend()
plt.grid(True)
plt.show()

## Sampling Distance Verification (Blank)

In [ ]:
%matplotlib inline
# Parameters from your model
alpha = 1.5
min_distance = 0.7
max_distance = max(all_distances) + 1  # To fit plot range

# Generate x values for PDF
x_vals = np.linspace(min_distance, max_distance, 1000)

# Compute Lévy PDF (unnormalized), then normalize
levy_pdf = x_vals ** -(1 + alpha)
dx = x_vals[1] - x_vals[0]
levy_pdf /= np.sum(levy_pdf * dx)  # Normalize to make it a valid PDF

# Plot
%matplotlib inline
if all_distances:
    plt.hist(
        all_distances,
        bins=50,
        alpha=0.7,
        density=True,
        label="Sampled Distances",
        color="green"
    )
    plt.plot(x_vals, levy_pdf, label=f"Theoretical Lévy PDF (α={alpha})", color='blue', linewidth=2)
    plt.axvline(x=3, color='red', linestyle='--', linewidth=2, label='Long jump threshold (x = 3)')

    plt.xlim(0, max_distance)
    plt.title("Distribution of Bee Jump Distances (Truncated at x=20)")
    plt.xlabel("Jump Distance (meters)")
    plt.ylabel("Density")
    plt.grid(True)
    plt.legend()
    plt.show()
else:
    print("No jump distances recorded — skipping plot.")

## 2. Visualize Flight Path (Blank)

In [ ]:
%matplotlib inline

import pickle
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

def load_dataframe(filename):
    """Loads the dataframe from a pickle file."""
    with open(filename, "rb") as f:
        results = pickle.load(f)
    df = pd.DataFrame(results)
    return df.reset_index()


def vis_bees_path(df, bee_ids, start_time_str="10:00", duration_minutes=10, x_buffer=5):
    """
    Plot flight paths of selected bees with direction arrows,
    including the first beehive and relevant plant locations.

    Parameters:
        df (pd.DataFrame): Simulation dataframe.
        bee_ids (list of int): List of bee AgentIDs to visualize.
        start_time_str (str): Start time in HH:MM format (24h clock starting from 10:00).
        duration_minutes (int): Duration in minutes to plot.
        x_buffer (float): Extra range added to max x for plant inclusion.
        
    NOTE: Choose bees from beehive 1 (beeids 2 to 62). Need to add extra data "from hive" in model.py
    """
    def time_str_to_step(time_str):
        hh, mm = map(int, time_str.split(":"))
        return (hh - 10) * 3600 + mm * 60                                       

    start_step = time_str_to_step(start_time_str)
    end_step = start_step + duration_minutes * 60

    # Filter bees
    bee_df = df[(df["Agent Type"] == "bee") &
                (df["Step"] >= start_step) &
                (df["Step"] < end_step) &
                (df["AgentID"].isin(bee_ids))].copy()
    
    if bee_df.empty:
        print("No bee data found for the given IDs and time window.")
        return

    # Extract bee positions
    bee_df[['x', 'y']] = bee_df['Position'].apply(pd.Series)

    # Get x-range from bee paths
    min_x = bee_df['x'].min()
    max_x = bee_df['x'].max()

    # Prepare figure
    plt.figure(figsize=(10, 6))
    cmap = plt.colormaps.get_cmap("tab10").resampled(len(bee_ids))

    # Plot bee flight paths
    for idx, bee_id in enumerate(bee_ids):
        track = bee_df[bee_df['AgentID'] == bee_id].sort_values(by="Step")
        x = track['x'].values
        y = track['y'].values
        u = x[1:] - x[:-1]
        v = y[1:] - y[:-1]

        plt.plot(x, y, label=f"Bee {bee_id}", color=cmap(idx), linewidth=1.5, alpha=0.7)
        plt.quiver(x[:-1], y[:-1], u, v, angles='xy', scale_units='xy', scale=1,
                   width=0.0025, headwidth=3, headlength=4, color=cmap(idx))

    # Plot the first beehive at Step 0
    beehive_df = df[(df["Agent Type"] == "beehive") & (df["Step"] == 0)].copy()
    if not beehive_df.empty:
        beehive_df[['x', 'y']] = beehive_df['Position'].apply(pd.Series)
        beehive_df = beehive_df.sort_values(by="unique_id").head(1)  # First hive only
        for _, hive in beehive_df.iterrows():
            plt.scatter(hive['x'], hive['y'], marker='^', c='brown', edgecolors='black', s=100, zorder=3)
            plt.text(hive['x'], hive['y'] + 0.5, f"Hive {hive['unique_id']}",
                     ha='center', va='bottom', fontsize=9, color='black')

    # Plot filtered plant locations at Step 0
    plant_df = df[(df["Agent Type"] == "plant") & (df["Step"] == 0)].copy()
    if not plant_df.empty:
        plant_df[['x', 'y']] = plant_df['Position'].apply(pd.Series)
        filtered_plants = plant_df[(plant_df['x'] >= min_x) & (plant_df['x'] <= max_x + x_buffer)]
        plt.scatter(filtered_plants['x'], filtered_plants['y'], marker='o', c='green', edgecolors='black', s=40, alpha=0.6, label='Plant', zorder=1)

    # Final formatting
    plt.title(f"Bee Flight Paths from {start_time_str} for {duration_minutes} min")
    plt.xlabel("X Position")
    plt.ylabel("Y Position")
    plt.grid(True)
    plt.axis("equal")
    plt.legend()
    plt.show()


In [ ]:
df = load_dataframe("sim_results_my_behavior_seed_0_blank.pkl")

In [ ]:
%matplotlib inline
# Select bee IDs to visualize
vis_bees_path(df, bee_ids=[5], start_time_str="10:00", duration_minutes=10)